# Leaflet cluster map of talk locations

Assuming you are working in a Linux or Windows Subsystem for Linux environment, you may need to install some dependencies. Assuming a clean installation, the following will be needed:

```bash
sudo apt install jupyter
sudo apt install python3-pip
pip install python-frontmatter getorg --upgrade
```

After which you can run this from the `_talks/` directory, via:

```bash
 jupyter nbconvert --to notebook --execute talkmap.ipynb --output talkmap_out.ipynb
```
 
The `_talks/` directory contains `.md` files of all your talks. This scrapes the location YAML field from each `.md` file, geolocates it with `geopy/Nominatim`, and uses the `getorg` library to output data, HTML, and Javascript for a standalone cluster map.

In [1]:
# Start by installing the dependencies
!pip install python-frontmatter getorg --upgrade  # needs Python >= 3.10 (python-frontmatter uses typing.TypeGuard)
import frontmatter
import glob
import getorg
from geopy import Nominatim
from geopy.exc import GeocoderTimedOut

Iywidgets and ipyleaflet support disabled. You must be in a Jupyter notebook to use this feature.
Error raised:
No module named 'ipyleaflet'
Check that you have enabled ipyleaflet in Jupyter with:
    jupyter nbextension enable --py ipyleaflet


In [2]:
# Collect the Markdown files (recursive: talks live in per-year subdirectories)
g = glob.glob("_talks/**/*.md", recursive=True)

In [3]:
# Set the default timeout, in seconds
TIMEOUT = 5

# Prepare to geolocate
geocoder = Nominatim(user_agent="academicpages.github.io")
location_dict = {}
location = ""
permalink = ""
title = ""

In the event that this times out with an error, double check to make sure that the location is can be properly geolocated.

In [4]:
# Perform geolocation
for file in g:
    # Read the file
    data = frontmatter.load(file)
    data = data.to_dict()

    # Press on if the location is not present
    if 'location' not in data:
        continue

    # Prepare the description
    title = data['title'].strip()
    venue = data['venue'].strip()
    location = data['location'].strip()
    description = f"{title}<br />{venue}; {location}"

    # Geocode the location and report the status
    try:
        location_dict[description] = geocoder.geocode(location, timeout=TIMEOUT)
        print(description, location_dict[description])
    except ValueError as ex:
        print(f"Error: geocode failed on input {location} with message {ex}")
    except GeocoderTimedOut as ex:
        print(f"Error: geocode timed out on input {location} with message {ex}")
    except Exception as ex:
        print(f"An unhandled exception occurred while processing input {location} with message {ex}")

Entangled Interlocked Diamond-Like (Diamondiynes) Lattices<br />EOSBF 2026 — Encontro de Outono da Sociedade Brasileira de Física; Cuiabá, Brazil Cuiabá, Mato Grosso, Região Centro-Oeste, Brasil
A Complete Optoelectronic and Thermodynamic Characterization for MoWSe2 Alloy: A Data-Driven Workflow Approach<br />XXIII B-MRS Meeting; Salvador, Brazil Salvador, Bahia, Região Nordeste, Brasil
Computational Materials Discovery: From Monolayer Noble Metals to Entangled Carbon Allotropes<br />Graduate Program in Nanoscience and Advanced Materials, UFABC; Santo André, Brazil Santo André, São Paulo, Região Sudeste, Brasil
Realistic multibands k.p from hybrid-DFT<br />2° ICP Workshop on Quantum and Statistical Physics, University of Brasilia; Brasilia - DF, Brazil Banco Central do Brasil, Bloco B, SBS Quadra 3, Setor Bancário Sul, Asa Sul, Brasília, Plano Piloto, Distrito Federal, Região Centro-Oeste, 70074-900, Brasil


In [5]:
# Save the map data only. Deliberately NOT calling
# getorg.orgmap.output_html_cluster_map() here: it also (re)writes talkmap/map.html
# and talkmap/leaflet_dist/* from templates bundled inside the getorg package,
# which are pinned to a 2012-2013 Leaflet.markercluster incompatible with the
# modern Leaflet loaded in talkmap/map.html (markers silently failed to render).
# talkmap/map.html and talkmap/leaflet_dist/ are hand-maintained instead.
getorg.orgmap.location_dict_to_jsvar(location_dict, "talkmap/org-locations.js", hashed_usernames=False)

'Written to talkmap/org-locations.js'